# Gaussian-linear task with SNPE-C

Train a small SNPE-C estimator, sample its posterior at held-out observations, and run SBC/TARP using the common diagnostics API.

In [1]:
from pathlib import Path
import sys
import torch
import yaml
from sbi.inference import SNPE_C, simulate_for_sbi

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT / "code" if (ROOT / "code" / "src").exists() else ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from sfmpe.tasks.Gaussian import GaussianTask
from sfmpe.utils.metrics import check_sbc, check_tarp, run_sbc, run_tarp

with (ROOT / "configs" / "gaussian.yaml").open(encoding="utf-8") as stream:
    task = GaussianTask(yaml.safe_load(stream))
torch.manual_seed(41)

In [2]:
inference = SNPE_C(
    prior=task.prior,
    density_estimator="maf",
    show_progress_bars=False,
)
theta_train, x_train = simulate_for_sbi(
    task.simulate,
    proposal=task.prior,
    num_simulations=512,
    simulation_batch_size=64,
    show_progress_bar=False,
)
density_estimator = inference.append_simulations(
    theta_train, x_train
).train(
    max_num_epochs=20,
    stop_after_epochs=8,
    training_batch_size=64,
    show_train_summary=False,
)
posterior = inference.build_posterior(density_estimator)

Running 512 simulations.:   0%|          | 0/512 [00:00<?, ?it/s]

 Neural network successfully converged after 19 epochs.

In [3]:
num_observations = 10
num_posterior_samples = 64
thetas, observations = task.simulate_dataset((num_observations,))
posterior_samples = torch.stack(
    [
        posterior.sample(
            (num_posterior_samples,),
            x=task.summarize(observation),
            show_progress_bars=False,
        )
        for observation in observations
    ],
    dim=1,
)

ranks, dap_samples = run_sbc(
    thetas, posterior_samples, num_posterior_samples=num_posterior_samples
)
sbc_checks = check_sbc(
    ranks, task.sample_prior((num_observations,)), dap_samples,
    num_posterior_samples=num_posterior_samples,
)
ecp, alpha = run_tarp(thetas, posterior_samples, num_bins=5)
tarp_checks = check_tarp(ecp, alpha)
assert posterior_samples.shape == (
    num_posterior_samples, num_observations, task.theta_dim
)
assert torch.isfinite(ecp).all()
print("SBC KS p-values:", sbc_checks["ks_pvals"])
print("TARP (area-to-curve, KS p-value):", tarp_checks)

SBC KS p-values: tensor([0.9065, 0.1472])
TARP (area-to-curve, KS p-value): (0.06539062410593033, 0.9307359307359307)
